## 1) Importar bibliotecas

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from pathlib import Path
import keras_tuner as kt
import tensorflow as tf
import polars as pl
import numpy as np

## 2) Carregar dados

In [ ]:
mnist = tf.keras.datasets.mnist

mnist = mnist.load_data()

### 2.1) Separar dados em treino e teste

In [ ]:
(X_train, y_train), (X_test, y_test) = mnist

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

### 2.2) Avaliar desbalanceamento

<p>Aqui, por sorte, podemos assumir que os targets de treinamento estão balanceados.</p>

In [ ]:
target_distribution_df = pl.DataFrame(data = {"target": y_train}).group_by("target").agg(
    pl.col("target").len().alias("total"),
).with_columns(
    (pl.col("total")/pl.col("total").sum()).alias("percentage")
).sort(by = "target")

target_distribution_df

In [ ]:
fig, ax = plt.subplots()

bar_plot = ax.bar(
    x = target_distribution_df["target"],
    height = target_distribution_df["percentage"],
)

ax.bar_label(
    container = bar_plot,
    labels = [
        f"{np.round(a = percentage*100, decimals = 2)}%" for percentage in target_distribution_df["percentage"]
    ],
)

ax.spines[["top", "right"]].set_visible(False)
ax.set_xticks(ticks = np.arange(0,10, 1))
ax.set_title("Relative target distribution", fontsize = 16)

plt.show()

### 2.3) Mostrando imagens

In [ ]:
fig, axs = plt.subplots(
    nrows = 5,
    ncols = 8,
    figsize = (20, 12.5),
    gridspec_kw = {
        "hspace": 0.4,
        "wspace": 0.4
    }
)

for i, ax in enumerate(axs.flatten()):
    ax.imshow(X_train[i], cmap = "gray")
    ax.set_title(f"Número: {y_train[i]}", fontsize = 16)
    ax.spines[["top", "left", "bottom", "right"]].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

plt.show()

### 2.4) Mostrando números detalhadamente

In [ ]:
fig, ax = plt.subplots(
    figsize = (12, 12)
)

order = np.random.randint(low = 0, high = len(X_train))
image = X_train[order]/255

ax.imshow(
    X = image,
    cmap = "gray"
)

for y, y_axis in enumerate(image):
    for x, x_axis in enumerate(y_axis):
        value = image[y][x]
        ax.annotate(
            text = f"{value:.2f}",
            xy = (x, y),
            color = "white" if value < 0.5 else "black",
            verticalalignment = "center",
            horizontalalignment = "center",
            fontsize = 8
        )

ax.set_xticks([])
ax.set_yticks([])
ax.set_title(f"Foto: {order} | Número: {y_train[order]}", fontsize = 16)

plt.show()

## 3) Criar modelo

In [ ]:
X_test.shape

### 3.1) Funções de treinamento e teste:

#### 3.1.1) Criando <code>EarlyStopping</code>:

In [ ]:
model_early_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta = 1E-4,
    patience = 5,
    verbose = 1,
)

#### 3.1.2) Função de avaliação de treinamento <code>plot_trainning</code>:

In [ ]:
def plot_trainning(
    epoch: list,
    loss: list,
    val_loss: list,
):

    fig, ax = plt.subplots()

    ax.plot(
        epoch,
        loss,
        label = "loss"
    )

    ax.plot(
        epoch,
        val_loss,
        label = "val_loss"
    )
    ax.set_title("Evolução da loss segundo treinamento.")


    plt.legend()
    plt.show()

#### 3.1.3) Função de validação: <code>plot_matrix</code>:

In [ ]:
def plot_matrix(
    y_true: list[int | float],
    y_pred: list[int | float],
) -> None:
    
    fig, ax = plt.subplots()

    plot_matrix_display = ConfusionMatrixDisplay(
        confusion_matrix = confusion_matrix(
            y_true = y_true,
            y_pred = y_pred,
        ),
    )

    plot_matrix_display.plot(
        ax = ax,
    )

    plt.show()

### 3.2) Primeiro modelo:

#### 3.2.1) Definindo modelo

In [ ]:
model1 = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape = (28, 28)), # Determina leitura para (28, 28)
    tf.keras.layers.Rescaling(scale = 1./255), # Multiplica valores por 1/255
    tf.keras.layers.Flatten(), # Transforma n dim array em 1 dim
    tf.keras.layers.Dense(units = 128, activation = "relu"),
    tf.keras.layers.Dropout(0.2), # Tentar evitar overfitting desligando 20% dos neurônios
    tf.keras.layers.Dense(units = 128, activation = "relu"),
    tf.keras.layers.Dropout(0.2), # Tentar evitar overfitting desligando 20% dos neurônios
    tf.keras.layers.Dense(units = 10, activation = "softmax"),
])

model1.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate = 1E-4),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.keras.metrics.SparseCategoricalAccuracy]
)

model1.summary()

#### 3.2.2) Treinando modelo

In [ ]:
history1 = model1.fit(
    x = X_train,
    y = y_train,
    batch_size = 64,
    epochs = 100,
    shuffle = True,
    validation_split = 0.2,
    callbacks = [model_early_stopping]
)

#### 3.2.3) Avaliando qualidade do treinamento

In [ ]:
total_predictions = 1000

In [ ]:
plot_trainning(
    epoch = history1.epoch,
    loss = history1.history["loss"],
    val_loss = history1.history["val_loss"],
)

prediction_argmax = [
    model1.predict(
        tf.expand_dims(
            input = X_test[i],
            axis = 0
        ),
        verbose = False
    ).argmax() for i in range(total_predictions)
]

plot_matrix(
    y_true = y_test[:total_predictions],
    y_pred = prediction_argmax,
)

#### 3.2.4) Avaliando minhas próprias imagens

In [ ]:
fig, axs = plt.subplots(
    ncols = 5, nrows = 2,
    figsize = (10, 5),
    gridspec_kw = {
        "wspace": 0.5,
        "hspace": 0.4
    }
)

axs = axs.flatten()

for i, archive in enumerate(Path("./my_images").iterdir()):

    keras_img = tf.keras.utils.load_img(
            path = archive,
            color_mode = "grayscale",
            target_size = (28, 28) # Definimos a escala em que o modelo foi treinado
        )

    img_numpy = tf.keras.utils.img_to_array(
        img = keras_img
    )

    axs[i].imshow(
        (img_numpy),
        "gray"
    )

    prediction = model1.predict(
        tf.expand_dims(
            input = img_numpy,
            axis = 0
        ),
        verbose = False
    )

    color = "green" if prediction.argmax() == i else "red"

    axs[i].set_title(f"Número: {i}", color = color)
    axs[i].set_ylabel(f"Probab: {prediction.max():.2%}", color = color)
    axs[i].set_xlabel(f"Predição: {prediction.argmax()}", color = color)
    axs[i].set_xticks([])
    axs[i].set_yticks([])

plt.suptitle("Predições das minhas imagens", fontsize = 16)
plt.show()

### 3.3) Segundo modelo:

#### 3.3.1) Definindo modelo

In [ ]:
model2 = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape = (28, 28, 1)),
    tf.keras.layers.Rescaling(scale = 1./255),

    tf.keras.layers.RandomTranslation(height_factor = 0.15, width_factor = 0.15),
    tf.keras.layers.RandomZoom(height_factor = 0.15, width_factor = 0.15),
    tf.keras.layers.RandomRotation(factor = 0.15),

    tf.keras.layers.Conv2D(filters = 2**6, activation = "relu", kernel_size = (3, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(rate = 0.2),
    tf.keras.layers.Conv2D(filters = 2**6, activation = "relu", kernel_size = (3, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Dropout(rate = 0.2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(units = 10, activation = "softmax")
])

model2.compile(
    optimizer = tf.keras.optimizers.Adam(1E-4),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics = [tf.keras.metrics.SparseCategoricalAccuracy],
)

model2.summary()

#### 3.3.2) Treinando modelo

In [ ]:
history2 = model2.fit(
    x = X_train,
    y = y_train,
    batch_size = 64,
    epochs = 100,
    shuffle = True,
    validation_split = 0.2,
    callbacks = [model_early_stopping]
)

#### 3.3.3) Avaliando qualidade do treinamento

In [ ]:
total_predictions = 1000

In [ ]:
plot_trainning(
    epoch = history2.epoch,
    loss = history2.history["loss"],
    val_loss = history2.history["val_loss"],
)

prediction_argmax = [
    model2.predict(
        tf.expand_dims(
            input = X_test[i],
            axis = 0
        ),
        verbose = False
    ).argmax() for i in range(total_predictions)
]

plot_matrix(
    y_true = y_test[:total_predictions],
    y_pred = prediction_argmax,
)

#### 3.3.4) Avaliando minhas próprias imagens

In [ ]:
fig, axs = plt.subplots(
    ncols = 5, nrows = 2,
    figsize = (10, 5),
    gridspec_kw = {
        "wspace": 0.5,
        "hspace": 0.4
    }
)

axs = axs.flatten()

for i, archive in enumerate(Path("./my_images").iterdir()):

    keras_img = tf.keras.utils.load_img(
            path = archive,
            color_mode = "grayscale",
            target_size = (28, 28) # Definimos a escala em que o modelo foi treinado
        )

    img_numpy = tf.keras.utils.img_to_array(
        img = keras_img
    )

    axs[i].imshow(
        (img_numpy),
        "gray"
    )

    prediction = model2.predict(
        tf.expand_dims(
            input = img_numpy,
            axis = 0
        ),
        verbose = False
    )

    color = "green" if prediction.argmax() == i else "red"

    axs[i].set_title(f"Número: {i}", color = color)
    axs[i].set_ylabel(f"Probab: {prediction.max():.2%}", color = color)
    axs[i].set_xlabel(f"Predição: {prediction.argmax()}", color = color)
    axs[i].set_xticks([])
    axs[i].set_yticks([])

plt.suptitle("Predições das minhas imagens", fontsize = 16)
plt.show()

### 3.4) Terceiro modelo:

#### 3.4.1) Definindo modelo

In [ ]:
def model_builder(hp):

    kernel_options = [(2, 2), (3, 3), (4, 4)]
    kernel_size = hp.Choice("Conv2D_kernel", values = [0, 1, 2])
    selected_kernel = kernel_options[kernel_size]

    model = tf.keras.Sequential([
        # Configurações de entrada:
        tf.keras.layers.InputLayer(shape = (28, 28, 1)),
        tf.keras.layers.Rescaling(scale = 1./255),

        # Configurações de aleatoriedade de treinamento:
        tf.keras.layers.RandomTranslation(height_factor = 0.15, width_factor = 0.15),
        tf.keras.layers.RandomZoom(height_factor = 0.15, width_factor = 0.15),
        tf.keras.layers.RandomRotation(factor = 0.15),

        # Configurações de filtros:
        ############################################
        tf.keras.layers.Conv2D(
            filters = hp.Int(
                "Conv2D_1",
                min_value = 16,
                max_value = 256,
                step = 2,
                sampling = "log",
            ),
            kernel_size = selected_kernel,
            activation = "relu"
        ),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(
            rate = hp.Choice(
                "Dropout_1",
                [
                    0.20, 0.25, 0.30
                ]
            )
        ),

        tf.keras.layers.Conv2D(
            filters = hp.Int(
                "Conv2D_2",
                min_value = 16,
                max_value = 256,
                step = 2,
                sampling = "log"
            ),
            kernel_size = selected_kernel,
            activation = "relu"
        ),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(
            rate = hp.Choice(
                "Dropout_2",
                [
                    0.20, 0.25, 0.30
                ]
            )
        ),
        ############################################

        # Configurações de saída:
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(units = 10, activation = "softmax")
    ])

    model.compile(
        optimizer = tf.keras.optimizers.Adam(1E-3),
        loss = tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics = [tf.keras.metrics.SparseCategoricalAccuracy()],
    )

    return model

In [ ]:
tuner = kt.Hyperband(
    hypermodel = model_builder,
    objective = "val_loss",
    max_epochs = 30,
    factor = 3,
    directory="C:/keras_tuner_models", # O caminho que você criou
    project_name="mnist",
    overwrite = False
)

#### 3.4.2) Treinando modelo

In [ ]:
tuner.search(
    x = X_train,
    y = y_train,
    batch_size = 32,
    validation_split = 0.2,
    callbacks = [model_early_stopping]
)

print("Procura de hiperparâmetros finalizada:")
print(tuner.get_best_hyperparameters)

#### 3.4.3) Reconhecendo modelo vencedor

<p>O <code>keras_tuner</code> automaticamente já "salva" o melhor parâmetro no diretório selecionado dentre vários outros modelos não vencedores. Para encontrá-lo, roda-se métodos <code>.get_best_hyperparameters</code> e <code>.get_best_models</code>.</p>

In [ ]:
best_paramns = tuner.get_best_hyperparameters()[0]

print(best_paramns.values)

In [ ]:
best_model = tuner.get_best_models()[0]

best_model.summary()

#### 3.4.4) Treinando modelo parametrizado

<p>Novamente, <code>keras_tuner</code>, se informado corretamente as mesmas propriedades de quando realizou o <code>.search</code>, vai perceber que não há variação alguma para ser testada caso não houver <code>.search</code>, e portanto obrigatoriamente irá disponibilizar a oportunidade de pegar o melhor modelo através de seu hiperparâmetro.</p>

<p><code>model_builder</code> pode reconstruir e devolver um model de acordo com as especificações de <code>.get_best_hyperparameters</code>, bastaria apenas treinar novamente o modelo.</p>

In [ ]:
hyper_paramn = kt.Hyperband(
    hypermodel=model_builder,
    objective="val_loss",
    max_epochs=30,
    directory="C:/keras_tuner_models",
    project_name="mnist",
    overwrite=False # IMPORTANTE: Se colocar True e sem querer inicializar .search, apagariam-se todo tempo de treino!
)

best_paramns = hyper_paramn.get_best_hyperparameters()[0]

#model_builder irá reconstruir o modelo passando hp = best_paramns
best_model = model_builder(best_paramns)

history3 = best_model.fit(
    x = X_train,
    y = y_train,
    batch_size = 64,
    epochs = 100,
    shuffle = True,
    validation_split = 0.2,
    callbacks = [model_early_stopping]
)

In [ ]:
total_predictions = 1_000

In [ ]:
plot_trainning(
    epoch = history3.epoch,
    loss = history3.history["loss"],
    val_loss = history3.history["val_loss"],
)

prediction_argmax = [
    best_model.predict(
        tf.expand_dims(
            input = X_test[i],
            axis = 0
        ),
        verbose = False
    ).argmax() for i in range(total_predictions)
]

plot_matrix(
    y_true = y_test[:total_predictions],
    y_pred = prediction_argmax,
)

#### 3.4.5) Avaliando qualidade do treinamento

In [ ]:
best_model.evaluate(
    X_test, y_test
)

In [ ]:
hyper_paramn.get_best_models()[0].evaluate(
    X_test, y_test
)

#### 3.4.6) Avaliando minhas próprias imagens

In [ ]:
fig, axs = plt.subplots(
    ncols = 5, nrows = 2,
    figsize = (10, 5),
    gridspec_kw = {
        "wspace": 0.5,
        "hspace": 0.4
    }
)

axs = axs.flatten()

for i, archive in enumerate(Path("./my_images").iterdir()):

    keras_img = tf.keras.utils.load_img(
            path = archive,
            color_mode = "grayscale",
            target_size = (28, 28) # Definimos a escala em que o modelo foi treinado
        )

    img_numpy = tf.keras.utils.img_to_array(
        img = keras_img
    )

    axs[i].imshow(
        (img_numpy),
        "gray"
    )

    prediction = best_model.predict(
        tf.expand_dims(
            input = img_numpy,
            axis = 0
        ),
        verbose = False
    )

    color = "green" if prediction.argmax() == i else "red"

    axs[i].set_title(f"Número: {i}", color = color)
    axs[i].set_ylabel(f"Probab: {prediction.max():.2%}", color = color)
    axs[i].set_xlabel(f"Predição: {prediction.argmax()}", color = color)
    axs[i].set_xticks([])
    axs[i].set_yticks([])

plt.suptitle("Predições das minhas imagens", fontsize = 16)
plt.show()